## Section 0 - Setup and Imports

In [ ]:
import sys
import os
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

project_root = Path().resolve().parents[1]
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import pandas as pd
import numpy as np

# LightGBM is a gradient boosting framework
# It builds an ensemble of decision trees where each tree corrects
# the errors of the previous one - this is what 'boosting' means
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

# SHAP explains individual predictions by computing each feature's
# contribution to the difference between the prediction and the
# average prediction across all samples
import shap

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from utils.helpers import set_style, save_figure, start_logging
set_style()
log = start_logging(project_root, '03_03_lightgbm_model')

print('All imports successful')
print(f'LightGBM version: {lgb.__version__}')
print(f'SHAP version: {shap.__version__}')

log('All imports successful')
log(f'LightGBM version: {lgb.__version__}')
log(f'SHAP version: {shap.__version__}')

---
## Section 1 - Load Data

In [ ]:
processed_path = project_root / 'data' / 'processed'

df_raw = pd.read_csv(
    processed_path / 'ts_features_weekly.csv',
    parse_dates=['week_start']
)
df = df_raw.copy()

print(f'Feature matrix loaded: {len(df):,} weeks, {len(df.columns)} columns')
print(f'Date range: {df["week_start"].min().date()} to {df["week_start"].max().date()}')
print(f'\nAll columns:')
for col in df.columns:
    print(f'  {col}')

log(f'Feature matrix loaded: {len(df):,} weeks, {len(df.columns)} columns')
log(f'Date range: {df["week_start"].min().date()} to {df["week_start"].max().date()}')

In [ ]:
# Filter to post-change-point data only
# Training on pre-COVID data would teach the model a revenue baseline
# of $15-25K per week. The post-COVID baseline is $50-80K per week.
# Mixing them produces a model that is wrong in both regimes.
CHANGE_POINT = '2020-03-02'
CUTOFF_DATE  = '2020-11-30'

df_post = df[df['week_start'] >= CHANGE_POINT].copy()

print(f'Post change-point weeks: {len(df_post):,}')
print(f'Date range: {df_post["week_start"].min().date()} to {df_post["week_start"].max().date()}')

log(f'Post change-point weeks: {len(df_post):,}')
log(f'Date range: {df_post["week_start"].min().date()} to {df_post["week_start"].max().date()}')

## Section 2 - Feature Selection and Train/Test Split

In [ ]:
# Features available at prediction time
# We exclude: order_count (correlated with revenue, not available before the week ends)
# We exclude: avg_price (same reason)
# We include: all lag and rolling features (computed from past weeks)
# We include: all calendar features (known in advance)
# We include: high_ticket_pct lag (we use the previous week's product mix)

FEATURE_COLS = [
    # Lag features
    'revenue_lag_1', 'revenue_lag_2', 'revenue_lag_4', 'revenue_lag_8',
    'order_count_lag_1', 'order_count_lag_2', 'order_count_lag_4',
    # Rolling features
    'revenue_roll_mean_4', 'revenue_roll_mean_8', 'revenue_roll_mean_13',
    'revenue_roll_std_4', 'revenue_roll_std_8',
    # Calendar features
    'month', 'quarter', 'week_of_year', 'is_weekend', 'is_month_end', 'is_q4',
    'month_sin', 'month_cos',
    # Regime and product mix
    'regime', 'high_ticket_pct'
]

TARGET = 'revenue'

# Keep only columns that exist in the dataframe
FEATURE_COLS = [f for f in FEATURE_COLS if f in df_post.columns]

print(f'Features selected: {len(FEATURE_COLS)}')
print(f'Features: {FEATURE_COLS}')

log(f'Features selected: {len(FEATURE_COLS)}')
log(f'Features: {FEATURE_COLS}')

In [ ]:
# Drop rows with NaN in any feature column
df_clean = df_post.dropna(subset=FEATURE_COLS + [TARGET]).copy()

# Temporal train/test split
# Train: post change-point up to November 2020
# Test: December 2020 onwards (the final weeks before dataset ends)
train = df_clean[df_clean['week_start'] <= CUTOFF_DATE].copy()
test  = df_clean[df_clean['week_start'] >  CUTOFF_DATE].copy()

X_train = train[FEATURE_COLS]
y_train = train[TARGET]
X_test  = test[FEATURE_COLS]
y_test  = test[TARGET]

print(f'Training weeks: {len(train):,} ({train["week_start"].min().date()} to {train["week_start"].max().date()})')
print(f'Test weeks:     {len(test):,} ({test["week_start"].min().date()} to {test["week_start"].max().date()})')
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

log(f'Training weeks: {len(train):,}')
log(f'Test weeks: {len(test):,}')

## Section 3 - Time Series Cross-Validation


In [ ]:
# TimeSeriesSplit with 3 folds
# With ~35 post-COVID training weeks, 3 folds gives us reasonable
# validation windows without making each fold too small to be meaningful

tscv = TimeSeriesSplit(n_splits=3)

lgb_params = {
    'objective':        'regression',
    'metric':           'mae',
    'num_leaves':       15,
    'learning_rate':    0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'min_data_in_leaf': 3,
    'verbose':          -1,
    'random_state':     42
}

# Why these parameters?
# num_leaves=15: small tree depth prevents overfitting on limited data
# learning_rate=0.05: slow learning rate requires more trees but generalises better
# feature_fraction=0.8: use 80% of features per tree - adds randomness and reduces overfitting
# min_data_in_leaf=3: minimum samples per leaf - critical with small datasets

cv_maes  = []
cv_mapes = []

print('Time series cross-validation results:')
print(f'{"Fold":<6} {"Train size":<12} {"Val size":<10} {"MAE":>12} {"MAPE":>10}')
print('-' * 54)

log('Time series cross-validation results:')

X_arr = X_train.values
y_arr = y_train.values

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_arr), 1):
    X_fold_train, X_fold_val = X_arr[train_idx], X_arr[val_idx]
    y_fold_train, y_fold_val = y_arr[train_idx], y_arr[val_idx]

    fold_model = lgb.LGBMRegressor(**lgb_params, n_estimators=200)
    fold_model.fit(
        X_fold_train, y_fold_train,
        eval_set=[(X_fold_val, y_fold_val)],
        callbacks=[lgb.early_stopping(20, verbose=False),
                   lgb.log_evaluation(-1)]
    )

    preds = fold_model.predict(X_fold_val)
    preds = np.clip(preds, 0, None) # pyright: ignore[reportArgumentType, reportCallIssue]

    mae  = mean_absolute_error(y_fold_val, preds) # pyright: ignore[reportArgumentType]
    mask = y_fold_val > 0 # pyright: ignore[reportOperatorIssue]
    mape = np.mean(np.abs((y_fold_val[mask] - preds[mask]) / y_fold_val[mask])) * 100

    cv_maes.append(mae)
    cv_mapes.append(mape)

    print(f'{fold:<6} {len(train_idx):<12} {len(val_idx):<10} ${mae:>10,.0f} {mape:>9.1f}%')
    log(f'Fold {fold}: MAE ${mae:,.0f}, MAPE {mape:.1f}%')

print()
print(f'Mean CV MAE:  ${np.mean(cv_maes):,.0f}')
print(f'Mean CV MAPE: {np.mean(cv_mapes):.1f}%')

log(f'Mean CV MAE: ${np.mean(cv_maes):,.0f}')
log(f'Mean CV MAPE: {np.mean(cv_mapes):.1f}%')

---
## Section 4 - Train Final Model

In [ ]:
# Train final model on full training set
# We use the same parameters validated by cross-validation
final_model = lgb.LGBMRegressor(**lgb_params, n_estimators=300)
final_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(30, verbose=False),
               lgb.log_evaluation(-1)]
)

print(f'Final model trained')
print(f'Best iteration: {final_model.best_iteration_}')

log(f'Final model trained')
log(f'Best iteration: {final_model.best_iteration_}')

# Generate test predictions
test_preds = np.clip(final_model.predict(X_test), 0, None) # pyright: ignore[reportArgumentType, reportCallIssue]
train_preds = np.clip(final_model.predict(X_train), 0, None) # pyright: ignore[reportArgumentType, reportCallIssue]

# Calculate metrics
def mape(actual, predicted):
    mask = actual > 0
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100

test_mae  = mean_absolute_error(y_test, test_preds)
test_mape = mape(y_test.values, test_preds)
test_rmse = np.sqrt(np.mean((y_test.values - test_preds) ** 2))

train_mae  = mean_absolute_error(y_train, train_preds)
train_mape = mape(y_train.values, train_preds)

print()
print('LightGBM test set performance')
print(f'  MAPE: {test_mape:.2f}%')
print(f'  MAE:  ${test_mae:,.0f} per week')
print(f'  RMSE: ${test_rmse:,.0f} per week')
print()
print('LightGBM training set performance')
print(f'  MAPE: {train_mape:.2f}%')
print(f'  MAE:  ${train_mae:,.0f} per week')

log('LightGBM test set performance')
log(f'  MAPE: {test_mape:.2f}%')
log(f'  MAE: ${test_mae:,.0f} per week')
log(f'  RMSE: ${test_rmse:,.0f} per week')
log('LightGBM training performance')
log(f'  MAPE: {train_mape:.2f}%')
log(f'  MAE: ${train_mae:,.0f} per week')

## Section 5 - SHAP Feature Importance

In [ ]:
# Calculate SHAP values on the training set
explainer   = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_train)

# Mean absolute SHAP value per feature - overall importance ranking
shap_importance = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'mean_shap':  np.abs(shap_values).mean(axis=0)
}).sort_values('mean_shap', ascending=False)

print('Feature importance by mean absolute SHAP value')
print(f'{"Rank":<6} {"Feature":<30} {"Mean |SHAP|":>14}')
print('-' * 52)
for i, row in shap_importance.iterrows():
    rank = shap_importance.index.get_loc(i) + 1 # pyright: ignore[reportOperatorIssue]
    print(f'{rank:<6} {row["feature"]:<30} ${row["mean_shap"]:>12,.0f}')

log('SHAP feature importance computed')
log(shap_importance.to_string())

---
## Section 6 - Visualisations

In [ ]:
# Chart 1: Forecast vs actual on test set
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(test['week_start'], y_test.values, # pyright: ignore[reportArgumentType]
        color='#2E75B6', linewidth=2, marker='o', markersize=5, label='Actual')
ax.plot(test['week_start'], test_preds,
        color='#ED7D31', linewidth=2, marker='s', markersize=5,
        linestyle='--', label='LightGBM forecast')

for i, (date, actual, pred) in enumerate(zip(
        test['week_start'], y_test.values, test_preds)):
    err = actual - pred
    color = '#C00000' if err < 0 else '#1E6B3C'
    ax.annotate(f'${err:+,.0f}',
                xy=(date, max(actual, pred)),
                xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=8, color=color)

ax.set_title(f'LightGBM forecast vs actual - test set\nMAPE: {test_mape:.1f}%   MAE: ${test_mae:,.0f}/week',
             fontsize=13, fontweight='medium')
ax.set_ylabel('Weekly revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=9)
plt.xticks(rotation=30, ha='right')

save_figure(fig, '03_03_lgbm_forecast_vs_actual.png')
plt.show()

In [ ]:
# Chart 2: SHAP summary bar chart - top 12 features
top_features = shap_importance.head(12)

fig, ax = plt.subplots(figsize=(11, 6))

colors = ['#1F4E79' if i < 3 else '#2E75B6' if i < 6 else '#70AD47'
          for i in range(len(top_features))]

bars = ax.barh(top_features['feature'][::-1],
               top_features['mean_shap'][::-1],
               color=colors[::-1], alpha=0.85)

for bar, val in zip(bars, top_features['mean_shap'][::-1]):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontsize=9)

ax.set_title('Feature importance - mean absolute SHAP value\n(higher = more influence on revenue predictions)',
             fontsize=13, fontweight='medium')
ax.set_xlabel('Mean absolute SHAP value (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

save_figure(fig, '03_03_shap_feature_importance.png')
plt.show()

In [ ]:
# Chart 3: SHAP beeswarm plot
# Each dot is one training week
# Position on x-axis shows SHAP value (impact on prediction)
# Colour shows feature value (red = high, blue = low)
# This shows both direction AND magnitude of each feature's effect

top_feature_names = shap_importance['feature'].head(10).tolist()
top_indices = [FEATURE_COLS.index(f) for f in top_feature_names]

# summary_plot creates its own figure internally in this version of SHAP
# We let it render then save via plt.gcf() to grab the current figure
shap.summary_plot(
    shap_values[:, top_indices],
    X_train.iloc[:, top_indices],
    feature_names=top_feature_names,
    show=False,
    plot_size=(11, 7)
)

fig = plt.gcf()
fig.suptitle('SHAP beeswarm plot - top 10 features\n'
             'Red = high feature value, Blue = low. '
             'Right = pushes prediction up, Left = pushes down',
             fontsize=11, fontweight='medium', y=1.01)

save_figure(fig, '03_03_shap_beeswarm.png')
plt.show()

In [ ]:
# Chart 4: Residuals analysis
train_residuals = y_train.values - train_preds

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(train['week_start'], train_residuals,
             color='#2E75B6', linewidth=1, alpha=0.8)
axes[0].axhline(0, color='#C00000', linestyle='--', linewidth=1)
axes[0].set_title('Residuals over time (should be random around zero)',
                  fontsize=11, fontweight='medium')
axes[0].set_ylabel('Residual (actual minus predicted)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')

axes[1].hist(train_residuals, bins=20,
             color='#2E75B6', alpha=0.8, edgecolor='white')
axes[1].axvline(0, color='#C00000', linestyle='--', linewidth=1.5, label='Zero')
axes[1].axvline(train_residuals.mean(), color='#ED7D31', linestyle='--',
                linewidth=1.5, label=f'Mean: ${train_residuals.mean():,.0f}')
axes[1].set_title('Residual distribution (should be centred near zero)',
                  fontsize=11, fontweight='medium')
axes[1].set_xlabel('Residual (USD)')
axes[1].legend(fontsize=9)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

fig.suptitle('LightGBM residuals analysis', fontsize=13, fontweight='medium')
save_figure(fig, '03_03_lgbm_residuals.png')
plt.show()

---
## Section 7 - Findings

In [ ]:
top_feature   = shap_importance.iloc[0]['feature']
top_shap_val  = shap_importance.iloc[0]['mean_shap']
second_feat   = shap_importance.iloc[1]['feature']
second_shap   = shap_importance.iloc[1]['mean_shap']

print('Notebook 3 Findings - LightGBM Model')
print()
print('Model configuration')
print(f'  Training period: post change-point only ({CHANGE_POINT} to {CUTOFF_DATE})')
print(f'  Algorithm: LightGBM gradient boosting regression')
print(f'  Validation: 3-fold time series cross-validation')
print(f'  Features used: {len(FEATURE_COLS)}')
print()
print('Cross-validation results - most reliable performance estimate')
print(f'  Fold 1 MAPE: {cv_mapes[0]:.1f}%')
print(f'  Fold 2 MAPE: {cv_mapes[1]:.1f}%')
print(f'  Fold 3 MAPE: {cv_mapes[2]:.1f}%')
print(f'  Mean CV MAPE: {np.mean(cv_mapes):.1f}%')
print(f'  Mean CV MAE: ${np.mean(cv_maes):,.0f} per week')
print()
print('Test set performance')
print(f'  MAPE: {test_mape:.2f}% - not a valid benchmark (see Finding 1)')
print(f'  MAE: ${test_mae:,.0f} per week')
print()
print('Training set performance')
print(f'  MAPE: {train_mape:.2f}% - signals overfitting (see Finding 2)')
print(f'  MAE: ${train_mae:,.0f} per week')
print()
print('Finding 1 - Test MAPE is invalid for the same reason as Prophet')
print('  The test period (Jan-Feb 2021) contains the same dataset')
print('  truncation identified in Notebook 2. Revenue collapsed from')
print('  $122K to $3.6K in 8 weeks due to pre-order orders with')
print('  future ship dates being excluded. Both models predicted')
print('  $50-75K per week based on legitimate training signal.')
print('  The test MAPE of 339% reflects bad test data, not bad models.')
print()
print('Finding 2 - Training MAPE of 0.83% indicates overfitting')
print('  With 40 training weeks and 22 features, the model has too')
print('  many degrees of freedom relative to data points. It is')
print('  memorising the training set rather than generalising.')
print('  Cross-validation MAPE of 11.6% is the honest estimate.')
print()
print('Finding 3 - Revenue volatility is the most important predictor')
print(f'  {top_feature} ranks first with SHAP value ${top_shap_val:,.0f}.')
print('  High recent revenue volatility predicts higher future revenue.')
print('  This makes sense for gaming hardware where product launch')
print('  spikes create high-variance periods that tend to sustain.')
print()
print('Finding 4 - Product mix matters more than revenue momentum')
print(f'  {second_feat} ranks second with SHAP value ${second_shap:,.0f}.')
print('  When high-ticket items dominate the weekly order mix,')
print('  the model predicts significantly higher revenue. Low')
print('  high-ticket weeks push predictions down by up to $8,000.')
print('  This confirms the Project 02 finding that product mix')
print('  is a key driver of CLV variation across weeks.')
print()
print('Finding 5 - Four features have zero predictive value')
print('  is_weekend, is_month_end, is_q4, and regime all have')
print('  SHAP values of zero. Day-of-week has no effect confirmed.')
print('  Once rolling averages are included, the regime flag adds')
print('  no marginal information - the rolling features already')
print('  capture which revenue regime the model is operating in.')
print()
print('Finding 6 - Cross-validation degradation is expected')
print('  MAPE increases from 8% to 15.9% across folds because')
print('  later folds validate on the high-volatility Christmas')
print('  and new year period which is harder to predict. This')
print('  is honest model behaviour, not a failure.')
print()
print('Comparison with Prophet for Notebook 4')
print(f'  LightGBM cross-validation MAPE: {np.mean(cv_mapes):.1f}%')
print(f'  Prophet training MAPE: 25.13%')
print(f'  On consistent methodology LightGBM outperforms Prophet.')
print(f'  Both models fail on the truncated test period.')
print()
print('Notebook 3 complete')
print(f'Figures saved to reports/figures (4 charts)')
print('Ready for Notebook 4: Model Comparison')


---
## Section 8 - Export

In [ ]:
# Save LightGBM predictions
lgbm_output = test[['week_start', TARGET]].copy()
lgbm_output['lgbm_pred'] = test_preds
lgbm_output['lgbm_error'] = lgbm_output[TARGET] - lgbm_output['lgbm_pred']

lgbm_path = processed_path / 'lgbm_forecast.csv'
lgbm_output.to_csv(lgbm_path, index=False)

print(f'Exported: lgbm_forecast.csv ({len(lgbm_output):,} rows)')

# Append LightGBM metrics to model_metrics.csv for Notebook 4
metrics_path = processed_path / 'model_metrics.csv'
existing     = pd.read_csv(metrics_path)

lgbm_metrics = pd.DataFrame([{
    'model':       'LightGBM',
    'mape':        round(test_mape, 4),
    'mae':         round(test_mae, 2),
    'rmse':        round(test_rmse, 2),
    'train_mape':  round(train_mape, 4),
    'test_weeks':  len(test),
    'train_weeks': len(train)
}])

metrics_combined = pd.concat([existing, lgbm_metrics], ignore_index=True)
metrics_combined.to_csv(metrics_path, index=False)

print(f'Updated: model_metrics.csv')
print(metrics_combined.to_string(index=False))